# Lab 1 · Prediction: final-exam marks
**Goal:** predict each student's final marks from what is known *before* the exam, and prove how good the prediction is.

**You will practise**
- Look at the data before modelling
- Lock a test set away first
- Start from a baseline
- Find and remove a leak
- Build a pipeline: impute, scale, one-hot
- Compare models with cross-validation
- Test once, report a range
- Read the mistakes, explain the model
- Save the model, check inputs before trusting it

**How to work:** 🧠 = write your guess first · ✓ = the notebook proves the result

## 0 · Setup
- One seed for everything
- Print library versions, so results can be reproduced

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import HuberRegressor, LinearRegression
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

SEED = 0
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "font.size": 12})
print("pandas", pd.__version__, "· scikit-learn", sklearn.__version__)

## 1 · Load the data
- 1,000 students
- Target: `final_marks`
- Missing file? Run `python make_data.py`

In [ ]:
df = pd.read_csv("data/students.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

### Check three things first
- Missing values
- Ranges (min, max)
- Categories and their counts

In [ ]:
print("missing per column:\n", df.isna().sum()[df.isna().sum() > 0], "\n")
print(df.describe().round(1).T[["mean", "min", "max"]], "\n")
print(df["stream"].value_counts().to_dict(), df["tuition"].value_counts().to_dict())

### Each clue against the target
- Is it a line, a curve, or nothing?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, col in zip(axes, ["study_hours", "attendance", "sleep_hours", "prev_marks"]):
    ax.scatter(df[col], df["final_marks"], s=6, alpha=0.4)
    ax.set_xlabel(col)
axes[0].set_ylabel("final_marks")
plt.tight_layout(); plt.show()

**What to notice**
- Study hours, previous marks: roughly straight lines up
- Sleep: a **hump**, best near 7.5 hours; a straight line can't fit it
- A few dots far **below** the cloud: students ill on exam day

## 2 · 🧠 Predict first
- Asha: science, 12 study hours/week, 92% attendance, sleeps 7.5 h, 70 last term, no tuition
- Write your guess before the model does

In [ ]:
my_guess_for_asha = 80

## 3 · Lock away a test set
- Put 20% aside **before** any modelling
- Use it **once**, at the very end
- Compare and tune on the other 80% only

In [ ]:
target = "final_marks"
X = df.drop(columns=[target, "student_id"])     # an ID is never a clue
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"train {len(X_train)} · test {len(X_test)} (locked)")

## 4 · Baseline
- Always predict the training average; every real model must beat it
- Score: **MAE** (mean absolute error) = average miss, in marks
- **5-fold cross-validation:** train on 4 parts, score the 5th, rotate, average

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

def cv_mae(model, X_=None):
    """Average MAE over 5 folds of the training set."""
    X_ = X_train if X_ is None else X_
    return -cross_val_score(model, X_, y_train, cv=cv, scoring="neg_mean_absolute_error").mean()

baseline = cv_mae(DummyRegressor(strategy="mean"))
print(f"baseline (always the average): MAE {baseline:.2f} marks")

## 5 · 🧠 A suspicious column: `grade`
- How good would a model using **only** `grade` look?

In [ ]:
grade_only = Pipeline([
    ("prep", ColumnTransformer([("g", OneHotEncoder(), ["grade"])])),
    ("model", LinearRegression()),
])
print(f"grade only: MAE {cv_mae(grade_only):.2f}  vs baseline {baseline:.2f}")

**Too good to be true**
- Ask: *does this column exist at prediction time?*
- `grade` is computed from the final marks, after the exam
- It is a **leak**: drop it

In [ ]:
X_train = X_train.drop(columns="grade")
X_test = X_test.drop(columns="grade")
assert "grade" not in X_train.columns
print("✓ leak removed; columns:", list(X_train.columns))

## 6 · Pipeline: preparation inside the model
- Numbers: fill missing with the median, then standardise
- Categories: one-hot `stream`; `tuition` → 0/1
- Inside a `Pipeline`, these steps learn from training folds only (no hidden leak) and run automatically at prediction time

In [ ]:
numeric = ["study_hours", "attendance", "sleep_hours", "prev_marks"]
categorical = ["stream", "tuition"]

def make_prep(sleep_curve=False):
    parts = [
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), categorical),
    ]
    if sleep_curve:   # a curved clue: sleep and sleep²
        parts.append(("sleep²", make_pipeline(SimpleImputer(strategy="median"), PolynomialFeatures(2, include_bias=False), StandardScaler()), ["sleep_hours"]))
    return ColumnTransformer(parts)

make_prep().fit(X_train).get_feature_names_out()

## 7 · 🧠 Compare models fairly
- Same folds, same data, same score
- Rank them before running

| Model | Idea |
|---|---|
| linear | a straight line in every clue |
| linear + sleep² | adds a curved clue for the sleep hump |
| Huber + sleep² | same, but outliers pull less |
| random forest | many trees, averaged |
| boosting | trees that fix each other's mistakes |

In [ ]:
my_ranking = ["huber + sleep²", "linear + sleep²", "linear", "boosting", "random forest"]

In [ ]:
candidates = {
    "linear": Pipeline([("prep", make_prep()), ("model", LinearRegression())]),
    "linear + sleep²": Pipeline([("prep", make_prep(True)), ("model", LinearRegression())]),
    "huber + sleep²": Pipeline([("prep", make_prep(True)), ("model", HuberRegressor(max_iter=1000))]),
    "random forest": Pipeline([("prep", make_prep()), ("model", RandomForestRegressor(300, random_state=SEED, n_jobs=-1))]),
    "boosting": Pipeline([("prep", make_prep()), ("model", HistGradientBoostingRegressor(random_state=SEED))]),
}
results = pd.Series({name: cv_mae(m) for name, m in candidates.items()}).sort_values()
results.loc["baseline"] = baseline
print(results.round(2).to_string())

In [ ]:
results.drop("baseline").plot.barh(color="#2F5FD0"); plt.axvline(baseline, c="grey", ls="--", label="baseline")
plt.xlabel("cross-validated MAE (lower is better)"); plt.legend(); plt.gca().invert_yaxis(); plt.show()
assert results.idxmin() == "huber + sleep²" and results["linear + sleep²"] < results["linear"] < baseline
print("✓ the curved clue and the outlier-proof loss win; fancy trees don't beat knowing the shape")

**Lesson**
- Knowing the shape of the problem (a hump, a few outliers) beat fancier models
- Try the simple, well-understood model first

## 8 · Final test: once
- Train on all 800 training students
- Score the 200 locked students **one time**
- Report a **range** (bootstrap), not just one number

In [ ]:
final_model = candidates["huber + sleep²"].fit(X_train, y_train)
pred = final_model.predict(X_test)
errors = np.abs(pred - y_test.to_numpy())

rng = np.random.default_rng(SEED)
boot = [errors[rng.integers(0, len(errors), len(errors))].mean() for _ in range(5000)]
low, high = np.percentile(boot, [2.5, 97.5])
test_baseline = np.abs(y_train.mean() - y_test).mean()
print(f"test MAE {errors.mean():.2f} marks (95% range {low:.2f}–{high:.2f}) · baseline {test_baseline:.2f}")

In [ ]:
assert high < test_baseline * 0.6
print(f"✓ proof: even the worst likely MAE ({high:.2f}) is far below the baseline ({test_baseline:.2f})")

## 9 · Read the mistakes
- Predicted vs real
- What do the worst misses have in common?

In [ ]:
plt.scatter(y_test, pred, s=10, alpha=0.6); plt.plot([0, 100], [0, 100], c="grey", ls="--")
plt.xlabel("real marks"); plt.ylabel("predicted"); plt.title("Above the line = we over-predicted"); plt.show()

worst = X_test.assign(real=y_test, predicted=pred.round(1), miss=(y_test - pred).round(1)).sort_values("miss").head(5)
worst

**What the mistakes say**
- Worst misses: real marks far **below** the prediction: the ill-on-exam-day students
- No clue in the data predicts that: unavoidable, not a bug
- The data has ±4 marks of random noise, so MAE ≈ 3.2 is the best possible

## 10 · Explain the model
- **Permutation importance:** shuffle one column, see how much the error grows
- Bigger growth = the model relies on that clue more

In [ ]:
imp = permutation_importance(final_model, X_test, y_test, n_repeats=10, random_state=SEED, scoring="neg_mean_absolute_error")
importance = pd.Series(imp.importances_mean, index=X_test.columns).sort_values()
importance.plot.barh(color="#A9520E"); plt.xlabel("MAE increase when shuffled (marks)"); plt.show()
assert importance.idxmax() == "study_hours"
print("✓ study hours matter most, then previous marks, then sleep")

## 11 · Save and use the model
- Save the **whole pipeline** (preparation + model) in one file
- New data then needs no manual steps

In [ ]:
Path("models").mkdir(exist_ok=True)
joblib.dump(final_model, "models/marks_model.joblib")

loaded = joblib.load("models/marks_model.joblib")
asha = pd.DataFrame([{"stream": "science", "study_hours": 12, "attendance": 92, "sleep_hours": 7.5,
                      "prev_marks": 70, "tuition": "no"}])
asha_pred = float(loaded.predict(asha)[0])
print(f"Asha: model {asha_pred:.1f} · your guess {my_guess_for_asha}")

In [ ]:
assert abs(asha_pred - 85.7) < 2     # the true rule (we made the data) gives 85.7
print("✓ the saved pipeline predicts Asha within 2 marks of the truth")

### 🧠 Messy input
- A missing value and a stream the model never saw
- Will it crash, or answer? Will the answer be good?

In [ ]:
messy = asha.assign(attendance=np.nan, stream="vocational")
messy_pred = float(loaded.predict(messy)[0])
print(f"Asha with messy input: {messy_pred:.1f} (clean: {asha_pred:.1f})")

**Runs ≠ right**
- It answers, but ≈ 20 marks lower: the unknown stream got no bonus
- In a real service, check inputs first: flag unknown categories and missing values

In [ ]:
def check_input(row):
    """Return a list of problems that make a prediction untrustworthy."""
    problems = [f"unknown {c}: {row[c]!r}" for c in categorical if row[c] not in set(X_train[c])]
    problems += [f"missing {c}" for c in numeric if pd.isna(row[c])]
    return problems

print("clean:", check_input(asha.iloc[0]) or "ok")
print("messy:", check_input(messy.iloc[0]))
assert check_input(asha.iloc[0]) == [] and len(check_input(messy.iloc[0])) == 2
print("✓ flag untrustworthy inputs before trusting the number")

## 12 · Your turn
- Remove sleep² from the Huber model: how much worse is CV MAE?
- Swap Huber for plain linear: compare the worst 5 misses
- Add `hours_x_tuition` = study_hours × (tuition == "yes"): does CV improve? (never check on the test set)

In [ ]:
# TODO

## ✅ Checklist for every prediction project
- [x] Looked at the data: missing values, ranges, shapes, outliers
- [x] Locked a test set before modelling; used it once
- [x] Started from a baseline
- [x] Hunted leaks: *does this column exist at prediction time?*
- [x] All preparation inside a pipeline
- [x] Compared models with the same CV folds and score
- [x] Reported the test score with a range
- [x] Read the worst mistakes
- [x] Checked which clues matter
- [x] Saved the pipeline; check inputs before trusting a prediction